In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import pandas as pd

# Temporary: make src/ importable until the project is a proper package
sys.path.insert(0, str(Path.cwd().parent / "src"))

from load_raw import latest_coinbase_run, latest_tiingo_file, load_coinbase, load_tiingo
from validate import count_violations, hard_rule_violations, missing_calendar_days

In [ ]:
nvda = load_tiingo(latest_tiingo_file("NVDA"), "NVDA")
btc = load_coinbase(latest_coinbase_run("BTC-USD"), "BTC-USD")
df = pd.concat([nvda, btc], ignore_index=True)
df.info()

In [ ]:
df.groupby("symbol")[["adj_close", "adj_volume"]].describe()

In [ ]:
count_violations(hard_rule_violations(df))

In [ ]:
bad = df.copy()
bad.loc[0, "high"] = bad.loc[0, "low"] - 1          # high below low
bad.loc[1, "close"] = -5.0                          # negative price
bad.loc[2, "date"] = pd.Timestamp("2099-01-01")     # future date
bad = pd.concat([bad, bad.iloc[[3]]], ignore_index=True)  # duplicate row

count_violations(hard_rule_violations(bad))

In [ ]:
df = df.sort_values(["symbol", "date"], ignore_index=True)
df["daily_return"] = df.groupby("symbol")["adj_close"].pct_change()
df["raw_close_return"] = df.groupby("symbol")["close"].pct_change()

big_raw_moves = df["raw_close_return"].abs() > 0.5
df.loc[big_raw_moves, ["symbol", "date", "close", "adj_close", "raw_close_return", "daily_return"]]

In [ ]:
soft_violations = df["daily_return"].abs() > 0.5
print("Rows breaking the soft rule:", int(soft_violations.sum()))
df.groupby("symbol")["daily_return"].agg(["min", "max"])

## Any missing days from BTC??

**Question** Is there row for every calendar day?
**Keywords** pandas date range, values not in another -> 'pd.date_range' , 'isin'

In [ ]:
example_days = pd.date_range(start="2024-01-01", end="2024-01-05")
example_data = pd.Series(pd.to_datetime(["2024-01-01", "2024-01-02", "2024-01-04", "2024-01-05"]))

example_days[~example_days.isin(example_data)]


In [ ]:
btc_dates = df.loc[df["symbol"] == "BTC-USD", "date"]
missing_days = missing_calendar_days(btc_dates, btc_dates.min(), btc_dates.max())

print(f"BTC rows: {len(btc_dates)}, missing days: {len(missing_days)}")

In [ ]:
broken_dates = btc_dates[btc_dates != pd.Timestamp("2024-03-15")]
missing_calendar_days(broken_dates, btc_dates.min(), btc_dates.max())